In [1]:
import os
from pathlib import Path
import sys
from matplotlib import pyplot as plt
sys.path.append('../')  # srcディレクトリをimportできるようにする
from src.my_app.core.MyDataset.FineTuningDataset_v1_2 import FineTuningDataset_v1_2
from src.my_app.core.MyDataset import FineTuningDataset_v0
IMAGES_DIR = Path('../../kuzushiji-recognition/char_sep_datas')  # 画像ディレクトリ
GT_JSON_PATH = Path('../../kuzushiji-recognition/char_sep_datas/gt_json.json')    # アノテーションJSON (未使用でもロード例)

In [2]:
from torch.utils.data import DataLoader
test_doc_id_list = [
    '200021637',
    '100249371',
    '100249537',
    '200005598',
    '200014740',
    '200020019',
    '200021712',
    '200021869'
]
train_dataset = FineTuningDataset_v1_2(
    test_doc_id=test_doc_id_list,
    test_mode=False,
    images_dir=IMAGES_DIR, 
    json_path=GT_JSON_PATH,
    precompute_gt=True,
    target_width=300,
    )
test_dataset = FineTuningDataset_v1_2(
    test_doc_id=test_doc_id_list,
    test_mode=True,
    images_dir=IMAGES_DIR,
    json_path=GT_JSON_PATH,
    precompute_gt=True,
    target_width=300,
)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)


[FineTuningDataset_v1] precompute 完了 in 73423.2 ms  count=5204
[debug]: FineTuningDataset_v1_2 initialized.
[FineTuningDataset_v1] precompute 完了 in 13700.2 ms  count=947
[debug]: FineTuningDataset_v1_2 initialized.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _to_rgb01(t):
    if hasattr(t, "detach"):
        t = t.detach().cpu().numpy()
    if t.ndim == 3 and t.shape[0] in (1,3,4):
        if t.shape[0] == 1:
            t = np.repeat(t, 3, axis=0)
        t = np.transpose(t[:3], (1,2,0))
    t = t.astype(np.float32)
    t = np.clip(t, 0.0, 1.0)
    return t

def _to_chw_np(t):
    if hasattr(t, "detach"):
        t = t.detach().cpu().numpy()
    return t.astype(np.float32)

def _minmax(x, eps=1e-8):
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def overlay_first2_channels(img_chw, heatmap_chw, colors=((1.0,0.0,0.0),(0.0,1.0,0.0)), alphas=(0.45,0.45)):
    """
    先頭2チャネルのみ重ねる: ch0=colors[0], ch1=colors[1]
    img_chw: [3,H,W]
    heatmap_chw: [4,H,W]（少なくとも2チャネル必要）
    """
    img = _to_rgb01(img_chw)          # [H,W,3] in [0,1]
    heat = _to_chw_np(heatmap_chw)    # [C,H,W]
    assert heat.shape[0] >= 2, "ヒートマップは少なくとも2チャネル必要です。"

    h0 = _minmax(heat[0])
    h1 = _minmax(heat[1])

    cm0 = np.stack([h0*colors[0][0], h0*colors[0][1], h0*colors[0][2]], axis=-1)
    cm1 = np.stack([h1*colors[1][0], h1*colors[1][1], h1*colors[1][2]], axis=-1)

    out = img.copy()
    out = (1 - alphas[0]) * out + alphas[0] * cm0
    out = (1 - alphas[1]) * out + alphas[1] * cm1
    return np.clip(out, 0.0, 1.0)

# 使い方例（最初の1件だけ表示）
for batch in train_loader:
    img = batch[0][0]      # [3,H,W]
    heat = batch[1][0]     # [4,H,W]
    over = overlay_first2_channels(img, heat, colors=((1,0,0),(0,1,0)), alphas=(0.45,0.45))

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(_to_rgb01(img)); plt.title("image"); plt.axis('off')
    plt.subplot(1,2,2); plt.imshow(over);           plt.title("ch0(red)+ch1(green)"); plt.axis('off')
    plt.tight_layout(); plt.show()
    # break